In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
p = pathlib.Path.cwd()
for q in (p, *p.parents):
    s = q / "src" / "ftbp"   # <- change "ftbp" if you rename the package
    if s.exists():
        sys.path.insert(0, str(s.parent))  # add .../src
        break
else:
    raise RuntimeError("src/ftbp not found")

In [ ]:
import numpy as np
import pandas as pd
import itertools
from math import comb
from scipy.optimize import brentq
from scipy.stats import norm, cauchy, uniform
from ftbp.estimators import psi, psi_prime

In [ ]:
import numpy as np
from scipy.integrate import quad
from scipy.optimize import root_scalar

# Standard normal PDF
def phi(a):
    return np.exp(-a**2 / 2) / np.sqrt(2 * np.pi)

# Asymptotic variance V(c)
def asymptotic_variance(c, loss_type):
    # E[ψ²]
    num_integrand = lambda a: psi(a, c, loss_type)**2 * phi(a)
    num, _ = quad(num_integrand, -np.inf, np.inf)
    
    # E[ψ']
    den_integrand = lambda a: psi_prime(a, c, loss_type) * phi(a)
    den, _ = quad(den_integrand, -np.inf, np.inf)
    
    return num / den**2

# Target: 95% efficiency → V(c) = 1/0.95
target_variance = 1 / 0.95

# Solve: asymptotic_variance(c) - target_variance = 0
def equation(c, loss_type):
    return asymptotic_variance(c, loss_type) - target_variance


loss_type = 'huber'  # Change as needed
sol = root_scalar(equation, args=(loss_type,), bracket=[0.1, 10.0], method='brentq')  # Adjust bracket as needed
print('huber:')
print(f"Calibrated c = {sol.root:.4f}")


loss_type = 'logcosh'  # Change as needed
sol = root_scalar(equation, args=(loss_type,), bracket=[0.1, 10.0], method='brentq')  # Adjust bracket as needed
print('logcosh:')
print(f"Calibrated c = {sol.root:.4f}")


loss_type = 'concordant'  # Change as needed
sol = root_scalar(equation, args=(loss_type,), bracket=[0.1, 10.0], method='brentq')  # Adjust bracket as needed
print('concordant:')
print(f"Calibrated c = {sol.root:.4f}")

In [ ]:
# we want to get a grid of values for logcosh and concordant for our level BP
def equation2(x, c, delta, loss_type):
    return psi(x, delta, loss_type) - c * delta
loss_type ='logcosh'
delta = 1.2047  # from previous calibration
roots = []
for c in list(0.1 * np.arange(1, 12) - 0.2):
    brentq_sol = root_scalar(equation2, args=(c, delta, loss_type), bracket=[-3, 3], method='brentq')
    print(f'c: {c}, delta: {delta}, root: {brentq_sol.root}')
    roots.append(float(np.round(brentq_sol.root, 3)))
print('logcosh roots:', roots)

loss_type ='concordant'
delta = 1.4811  # from previous calibration
roots = []
for c in list(0.1 * np.arange(1, 12) - 0.2):
    brentq_sol = root_scalar(equation2, args=(c, delta, loss_type), bracket=[-10, 10000], method='brentq')
    print(f'c: {c}, delta: {delta}, root: {brentq_sol.root}')
    roots.append(float(np.round(brentq_sol.root, 3)))
print('concordant roots:', roots, sep=', ')

In [ ]:
psi(10000, 1.4811, 'concordant')  # should be